# Urban Mobility – Post‑processing & EDA
This notebook loads **tripinfo.xml** and **netstate.xml** results for the `base`, `peak`, and `event` scenarios generated, then derives key performance indicators and visualisations.

In [ ]:
# Activate environment variables (if running in JupyterLab terminal, run a shell cell)
import os, pathlib, glob
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import sumolib.xml as sx
OUTPUT_ROOT = pathlib.Path('../data/outputs')
assert OUTPUT_ROOT.exists(), 'Run simulations first (scripts/run_batch.py)'

## Helper functions

In [ ]:
def load_tripinfo_folder(folder: pathlib.Path) -> pd.DataFrame:
    frames = []
    for xml_path in folder.rglob('tripinfo.xml'):
        run      = xml_path.parent.name
        scenario = xml_path.parent.parent.name
        records  = [{**e.attrib, 'run': run, 'scenario': scenario}
                    for e in sx.parse_fast(xml_path, 'tripinfo')]
        if records:
            frames.append(pd.DataFrame(records))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def load_netstate_first_run(folder: pathlib.Path, scenario: str) -> pd.DataFrame:
    xml_candidates = sorted((folder / scenario).rglob('netstate.xml'))
    if not xml_candidates:
        return pd.DataFrame()
    xml_path = xml_candidates[0]
    rows = []
    for ts in sx.parse_fast(xml_path, 'timestep'):
        t = float(ts.attrib['time'])
        rows.extend({
            'time': t,
            'edge': edge.attrib['id'],
            'speed': float(edge.attrib.get('speed', 0.0)),
            'occupancy': float(edge.attrib.get('occupancy', 0.0)),
            'scenario': scenario,
        } for edge in ts)
    return pd.DataFrame(rows)

## Load data

In [ ]:
trip_df = load_tripinfo_folder(OUTPUT_ROOT)
if trip_df.empty:
    raise RuntimeError('tripinfo.xml files not found. Run simulations first.')

num_cols = ['depart', 'arrival', 'duration', 'routeLength',
            'waitingTime', 'departDelay', 'arrivalDelay']
for col in num_cols:
    trip_df[col] = pd.to_numeric(trip_df[col], errors='coerce')

trip_df.head()

## Key performance indicators (KPIs)

In [ ]:
kpis = (trip_df.groupby('scenario')
                 .agg(avg_travel_time=('duration', 'mean'),
                      pct95_travel_time=('duration', lambda x: x.quantile(0.95)),
                      avg_waiting_time=('waitingTime', 'mean'),
                      trips=('id', 'count'))
                 .round(2))
kpis

## Travel‑time histograms (log‑scale)

In [ ]:
for scn, grp in trip_df.groupby('scenario'):
    grp['duration'].plot.hist(bins=60, log=True, alpha=0.75)
    plt.title(f'Travel‑time distribution – {scn}')
    plt.xlabel('seconds'); plt.ylabel('frequency (log)')
    plt.show()

## Violin‑plot comparison

In [ ]:
sns.violinplot(data=trip_df, x='scenario', y='duration', inner='quartile')
plt.title('Travel‑time distribution comparison')
plt.ylabel('seconds')
plt.show()

## Departure load curves (5‑minute bins)

In [ ]:
trip_df['depart_bin'] = (trip_df['depart'] // 300).astype(int)
load_curve = (trip_df.groupby(['scenario', 'depart_bin']).size()
                      .rename('departures').reset_index())
pivot = load_curve.pivot(index='depart_bin', columns='scenario', values='departures').fillna(0)
pivot.plot.line()
plt.title('Departures per 5‑min interval')
plt.xlabel('simulation time bin (5 min)'); plt.ylabel('# vehicles')
plt.show()

## Slowest edges snapshot (optional)

In [ ]:
edge_tables = []
for scn in trip_df['scenario'].unique():
    net_df = load_netstate_first_run(OUTPUT_ROOT, scn)
    if net_df.empty:
        continue
    slow = (net_df.groupby('edge')['speed'].mean()
                     .nsmallest(10).reset_index().rename(columns={'speed':'mean_speed'}))
    slow['scenario'] = scn
    edge_tables.append(slow)
if edge_tables:
    pd.concat(edge_tables).set_index(['scenario', 'edge'])